# Small Model Supremacy — Full Training Pipeline

Fine-tuning Qwen2.5-1.5B with QLoRA (4-bit) to outperform frontier models on structured data extraction.

**Setup:** Kaggle → GPU T4 x2 → Run All → Sleep

**Expected runtime:** ~2-3 hours

In [ ]:
# Cell 1: Setup
!git clone https://github.com/BhavanaR006/small-model-supremacy.git
%cd small-model-supremacy
!pip install peft bitsandbytes accelerate transformers datasets --quiet

In [ ]:
# Cell 2: Load Model + Apply QLoRA
import json, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 3: Prepare Data (all 1500 examples)
train_data = [json.loads(l) for l in open("data/train.jsonl")]
val_data = [json.loads(l) for l in open("data/val.jsonl")]

def format_strict(ex):
    output_json = json.dumps(ex['expected_output'], ensure_ascii=False)
    text = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{ex['schema_id']}\".

Text: {ex['input_text']}<|im_end|>
<|im_start|>assistant
{output_json}<|im_end|>"""
    return {"text": text}

train_dataset = Dataset.from_list([format_strict(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_strict(ex) for ex in val_data])

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

train_tok = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

train_tok = train_tok.map(add_labels, batched=True)
val_tok = val_tok.map(add_labels, batched=True)
train_tok.set_format("torch")
val_tok.set_format("torch")

print(f"Ready: {len(train_tok)} train, {len(val_tok)} val")

In [ ]:
# Cell 4: Train (batch=2 to avoid OOM, grad_accum=8 for effective batch 16)
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=3,
    bf16=True,
    max_grad_norm=1.0,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Training started. Progress bar below. ~2-3 hours.")
trainer.train()
trainer.save_model("./checkpoints/final")
tokenizer.save_pretrained("./checkpoints/final")
print("\n✓ Training complete! Model saved.")

In [ ]:
# Cell 5: Test the model
model.eval()

def extract(text, schema_id):
    prompt = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{schema_id}\".

Text: {text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stop_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, eos_token_id=stop_token_id)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)
    try:
        start = raw.index('{')
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == '{': depth += 1
            elif raw[i] == '}': depth -= 1
            if depth == 0: return json.loads(raw[start:i+1])
    except:
        return raw

print("="*60)
print("TEST RESULTS")
print("="*60)

tests = [
    ("Dr. Sarah Chen presented quantum error correction at NeurIPS 2025 in Vancouver, Canada.", "conference_talk_simple"),
    ("Prof. Liu Wei discussed efficient transformers for edge deployment at ICML 2025 in Vienna, Austria.", "conference_talk_simple"),
    ("The EcoSmart PureBlend Protein Powder is priced at 34.99 GBP. Organic, vegan certified. New, limited stock. 20x10x10cm, 0.5kg. SKU: ES-9012-P44.", "product_listing_medium"),
    ("At the World AI Summit 2025 in Dubai, Dr. Amina Hassan delivered a keynote on federated learning for healthcare.", "conference_talk_simple"),
    ("Samsung Galaxy Buds Pro 3 with ANC and 360 Audio. 149.99 USD, 20% off. Refurbished, in stock. 7x5x3cm, 0.06kg. SKU: SG-8823-B11. Features: Active Noise Cancellation, Wireless charging, IPX7.", "product_listing_medium"),
]

for text, schema in tests:
    print(f"\nSchema: {schema}")
    print(f"Input: {text[:100]}..." if len(text) > 100 else f"Input: {text}")
    result = extract(text, schema)
    print(f"Output: {json.dumps(result, indent=2) if isinstance(result, dict) else result}")

In [ ]:
# Cell 6: Package model for download
import shutil
shutil.make_archive('/kaggle/working/model_final', 'zip', './checkpoints/final')
print("✓ Model zipped at /kaggle/working/model_final.zip")
print("Download from the Output tab on the right.")